In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.special import gamma
from numba import jit
from regression_common import ridge_pair, lasso_coefficients, sparse_indices, discrepancy_candidate
from regression_results import BootstrapRun
from regression_spectral import SpectralDesign
from regression_bootstrap import bootstrap_errors, validate_backend
from threadpoolctl import threadpool_limits


In [ ]:
def compute_sigma(Y, X, beta):
    sigma = np.linalg.norm(Y - X @ beta)
    return sigma / np.sqrt(Y.shape[0])


In [ ]:
def compute_ridge(c, X, Y):
    beta_r, _ = ridge_pair(X, Y, c / X.shape[0])
    return np.linalg.norm(beta_r - beta)

def ridge(X, Y, C):
    from regression_common import ridge_pair
    return ridge_pair(X, Y, C / X.shape[0])


In [ ]:

def compute_norminf_Ridge(seed):
    rng = np.random.default_rng(seed)
    e_Ridge = rng.normal(0, sigma_Ridge, n)
    Y_Ridge = np.dot(X, beta_Ridge) + e_Ridge
    e_Ridge_debias = rng.normal(0, sigma_Ridge_debias, n)
    Y_Ridge_debias = np.dot(X, beta_Ridge_debias) + e_Ridge_debias
    beta_ridgestar, beta_ridgestar_debias = ridge(X, Y_Ridge, best_c)
    beta_Ridgestar, beta_Ridgestar_debias = ridge(X, Y_Ridge_debias, best_c)
    beta_ridgestar[np.abs(beta_ridgestar) <= bn_Ridge] = 0
    beta_Ridgestar_debias[np.abs(beta_Ridgestar_debias) <= bn_Ridge_debias]=0
    norminf_Ridge = np.linalg.norm(beta_ridgestar - beta_Ridge, ord=np.inf)
    norminf_Ridge_debias = np.linalg.norm(beta_Ridgestar_debias - beta_Ridge_debias, ord=np.inf)
    return norminf_Ridge, norminf_Ridge_debias


In [ ]:
beta_names = [
    'beta_Ridge', 'beta_Ridge_debias',
]

variable_names = [
    'Ridge', 'Ridge_debias',
]


In [ ]:
n        = 1000
p        = 1000
I        = np.eye(p)
C        = 200000
m        = 100


i_ridge  = 0
i_ridge_debias = 0
Sigma_ridge = 0
Sigma_ridge_debias = 0
Inf_norm_ridge = 0
Inf_norm_ridge_debias = 0
Norm_ridge = 0
Norm_ridge_debias = 0

# Output controls do not change the estimators or parameter selection.
N = 1000
B = 500
confidence = 0.95
noise_sd = 2.0
seed_root = 20261006
output_root = 'outputs/regression-bootstrap-v2'
resume_from = None

# Execution controls only; statistical settings remain unchanged.
backend = 'cpu'
batch_size = 64
cpu_workers = 4
blas_threads = 8
gpu_devices = (0,)
backend_info = validate_backend(backend, gpu_devices=gpu_devices)


In [ ]:
# Keep the seeded covariance SVD independent of execution threads.
with threadpool_limits(limits=1):
    np.random.seed(24)
    cov_matrix = np.full((p, p), 0.5)
    np.fill_diagonal(cov_matrix, 2.0)

    mean_vector = np.zeros(p)
    X = np.random.multivariate_normal(mean_vector, cov_matrix, n)

    U, S, Vt = np.linalg.svd(X)

    beta = np.zeros(p)


    beta[0:5]   = 2.0   # βi = 2.0, i = 1, 2, 3, 4, 5
    beta[5:10]  = -2.0  # βi = -2.0, i = 6, 7, 8, 9, 10
    beta[10:15] = 1.0   # βi = 1.0, i = 11, 12, 13, 14, 15
    beta[15:20] = -1.0  # βi = -1.0, i = 16, 17, 18, 19, 20

    # Generate 20 random indices
    indices = sparse_indices(np.random, p, size=20)

    # Get the indices of non-zero elements in beta
    nonzero_indices = np.nonzero(beta)[0]
    random_beta = np.zeros(p)
    random_beta[indices] = beta[nonzero_indices]
    beta = random_beta
    spectral_design = SpectralDesign(X, svd=(U, S, Vt))


In [ ]:
run_config = {
    'method': 'Ridge', 'n': n, 'p': p, 'N': N, 'B': B,
    'sigma': noise_sd, 'confidence': confidence, 'seed_root': seed_root,
    'design_seed': 24, 'noise_model': 'normal',
    'largest_singular_value': float(S[0]),
    'execution': {'backend': backend, 'batch_size': batch_size,
                  'cpu_workers': cpu_workers, 'blas_threads': blas_threads,
                  'gpu_devices': list(gpu_devices)},
    'backend_info': backend_info,
    'preprocess_blas_threads': 1,
    'threshold_grid': [0.001, 0.001, 1.0],
    'threshold_percentile': 0,
    'method_settings': {'selection': 'oracle_grid', 'C': C, 'm': m, 'alpha_divisor': n},
}
with threadpool_limits(limits=blas_threads), BootstrapRun(run_config, X, beta, 'Bootstrapridge.ipynb',
                  output_root=output_root, resume_from=resume_from) as run:
    print('Run directory:', run.directory)
    for x in range(run.start_index, N):
        print(x)
        outer_rng = np.random.default_rng((seed_root, x, 0, 0))


        mean    = 0
        std_dev = noise_sd


        e = outer_rng.normal(mean, std_dev, n)
        norm_e = np.linalg.norm(e)
        Y = np.dot(X, beta) + e

        selection_settings = dict(run_config['method_settings'])
        beta_Ridge, beta_Ridge_debias, selected_parameter, selection_diagnostics = spectral_design.select(
            Y, 'Ridge', selection_settings, beta)
        best_c = selected_parameter * n
        selected_residual_norm = selection_diagnostics['residual_norm']
        sigma_Ridge = compute_sigma(Y, X, beta_Ridge)
        sigma_Ridge_debias = compute_sigma(Y, X, beta_Ridge_debias)

        result = []

        for i in range(1, 1001):
            bn = 0.001 * i

            norms = [bn]
            for var_name in beta_names:
                var_copy = globals()[var_name].copy()
                var_copy[np.abs(var_copy) <= bn] = 0
                norm = np.linalg.norm(var_copy - beta)
                norms.append(norm)
            result.append(norms)
        columns = ['bn'] + [f'norm_{name}' for name in beta_names]
        df = pd.DataFrame(result, columns=columns)

        results = []


        for column in df.columns[1:]:
            min_val = df[column].min()
            min_indices = np.where(df[column] == min_val)[0]

            if len(min_indices) > 1 and np.all(np.diff(min_indices) == 1):

                # avg_index = np.mean(min_indices)
                avg_index = min_indices[0]
            else:

                avg_index = min_indices[0]

            selected_bn = np.interp(avg_index, np.arange(len(df)), df['bn'].to_numpy())
            results.append([min_val, selected_bn])
            results_array = np.array(results)

        for i, name in enumerate(variable_names):
            globals()[f'bn_{name}'] = results_array[i, 1]
            # globals()[f'bn_{name}'] = 0.175

        for name in variable_names:
            beta_var = globals()[f'beta_{name}']
            bn_var = globals()[f'bn_{name}']
            beta_var[np.abs(beta_var) <= bn_var] = 0

        for name in variable_names:
            sigma = compute_sigma(Y, X, globals()[f'beta_{name}'])
            globals()[f'sigma_{name}'] = sigma

        Sigma_ridge += np.abs(sigma_Ridge**2 - std_dev**2)
        Sigma_ridge_debias += np.abs(sigma_Ridge_debias**2 - std_dev**2)
        Norm_ridge += np.linalg.norm(beta_Ridge - beta)
        Norm_ridge_debias += np.linalg.norm(beta_Ridge_debias - beta)

        norminf_Ridge_list = []
        norminf_Ridge_debias_list = []
        results = bootstrap_errors(
            spectral_design, 'Ridge', selected_parameter, selection_settings,
            np.column_stack([beta_Ridge, beta_Ridge_debias]),
            [sigma_Ridge, sigma_Ridge_debias], [bn_Ridge, bn_Ridge_debias],
            B, seed_root, x, backend=backend, batch_size=batch_size,
            cpu_workers=cpu_workers, blas_threads=blas_threads, gpu_devices=gpu_devices,
        )
        for norminf_Ridge, norminf_Ridge_debias in results:
            norminf_Ridge_list.append(norminf_Ridge)
            norminf_Ridge_debias_list.append(norminf_Ridge_debias)
        Ridge_upper = np.percentile(norminf_Ridge_list, 100 * confidence)
        Ridge_upper_debias = np.percentile(norminf_Ridge_debias_list, 100 * confidence)
        inf_ridge = np.linalg.norm(beta - beta_Ridge, ord=np.inf)
        inf_ridge_debias = np.linalg.norm(beta - beta_Ridge_debias, ord=np.inf)
        Inf_norm_ridge += inf_ridge
        Inf_norm_ridge_debias += inf_ridge_debias
        if inf_ridge <= Ridge_upper:
            i_ridge += 1
        if inf_ridge_debias <= Ridge_upper_debias:
            i_ridge_debias += 1

        run.record(
            x, selected_parameter=selected_parameter,
            thresholds=[bn_Ridge, bn_Ridge_debias],
            estimates=np.column_stack([beta_Ridge, beta_Ridge_debias]),
            sigma=[sigma_Ridge, sigma_Ridge_debias],
            bootstrap_errors=np.asarray(results),
            threshold_scan=df.to_numpy(), threshold_columns=list(df.columns),
            diagnostics={**selection_diagnostics, 'parameter_kind': 'alpha', 'selected_residual_norm': selected_residual_norm},
        )


In [ ]:
# Read all committed replications, including those restored from a checkpoint.
summary = run.summary
i_ridge = int(round(summary['coverage_original'] * N))
i_ridge_debias = int(round(summary['coverage_debiased'] * N))
Sigma_ridge = summary['variance_absolute_error_original'] * N
Sigma_ridge_debias = summary['variance_absolute_error_debiased'] * N
Inf_norm_ridge = summary['linf_original'] * N
Inf_norm_ridge_debias = summary['linf_debiased'] * N
Norm_ridge = summary['l2_original'] * N
Norm_ridge_debias = summary['l2_debiased'] * N
print(pd.Series(summary, name='Ridge'))
print('Saved results:', run.directory)
